In [ ]:
# 1. Установка (один раз)
# !pip install -q vllm==0.6.3.post1 faiss-gpu sentence-transformers langchain-community wikipedia tqdm

import wikipedia
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

# ========================== НАСТРОЙКИ ==========================
# Выбери модель (все работают с vLLM + AWQ/GPTQ/4bit):
# Самые быстрые и умные на 2025:
MODEL_NAME = "unsloth/Qwen3-32B-Instruct-bnb-4bit"      # ~22 ГБ VRAM, супер быстро с Unsloth ядром
# MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct-AWQ"         # если хочешь чистый AWQ
# MODEL_NAME = "unsloth/Qwen3-8B-Instruct-bnb-4bit"    # если мало VRAM

MAX_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.95
BATCH_SIZE_EMB = 64
RETRIEVE_K = 5

# ========================== LLM ==========================
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,        # 2–4 если несколько GPU
    max_model_len=32768,
    gpu_memory_utilization=0.95,
    quantization="bitsandbytes" if "bnb" in MODEL_NAME else None,  # автоматически для unsloth-bnb
    enforce_eager=True,            # важно для Unsloth моделей!
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    stop=["</s>", "<|endoftext|>"]  # на всякий случай
)

# ========================== Эмбеддер ==========================
embedder = SentenceTransformer(
    "intfloat/multilingual-e5-large-instruct",  # в 2025 это лучший open-source мультиязычный
    device="cuda"
)

# ========================== Данные (Википедия RU) ==========================
wikipedia.set_lang("ru")
page = wikipedia.page("Нью-Йорк")
text = page.content

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
docs = splitter.split_text(text)
print(f"Кусков текста: {len(docs)}")

# ========================== FAISS индекс ==========================
print("Эмбеддим чанки...")
embeddings = []
for i in tqdm(range(0, len(docs), BATCH_SIZE_EMB)):
    batch = docs[i:i+BATCH_SIZE_EMB]
    emb = embedder.encode(batch, normalize_embeddings=True, batch_size=BATCH_SIZE_EMB)
    embeddings.append(emb)

embeddings = np.vstack(embeddings).astype(np.float32)
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)  # Inner Product = cosine если нормализованы
faiss.normalize_L2(embeddings)
index.add(embeddings)
print(f"FAISS готов, чанков: {index.ntotal}")

# ========================== RAG функции ==========================
def retrieve(query: str, k: int = RETRIEVE_K):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    D, I = index.search(q_emb, k)
    return [docs[i] for i in I[0]]

def ask_rag_batch(queries):
    # 1. Retrieval батчем
    q_embs = embedder.encode(queries, normalize_embeddings=True).astype(np.float32)
    D, I = index.search(q_embs, RETRIEVE_K)

    # 2. Формируем контексты и промпты
    prompts = []
    for query, indices in zip(queries, I):
        context = "\n\n".join([docs[i] for i in indices])
        prompt = f"""Используй ТОЛЬКО предоставленный контекст. Отвечай кратко и точно.

Контекст:
{context}

Вопрос: {query}
Ответ:"""
        prompts.append(prompt)

    # 3. Генерация через vLLM
    outputs = llm.generate(prompts, sampling_params)

    # 4. Возврат результатов
    results = []
    for q, out in zip(queries, outputs):
        answer = out.outputs[0].text.strip()
        results.append({"question": q, "answer": answer})
    return results

# ========================== Тест на твоих 30 вопросах ==========================
questions = [
    "Как называется главный остров, на котором расположена большая часть Нью-Йорка?",
    "Какой район (боро) является самым густонаселенным в Нью-Йорке?",
    "Какое название носил Нью-Йорк, когда он был голландской колонией?",
    "В каком году было завершено строительство Эмпайр-стейт-билдинг?",
    "Сколько всего районов (боро) входит в состав Нью-Йорка? Назовите их.",
    # ... и все остальные 25 вопросов
]

# Батч по 8–16 вопросов — оптимально для 4090 + Qwen3-32B
batch_size = 12
results = []
for i in range(0, len(questions), batch_size):
    batch = questions[i:i+batch_size]
    results.extend(ask_rag_batch(batch))

# Вывод
for r in results:
    print(f"Q: {r['question']}")
    print(f"A: {r['answer']}\n{'─'*50}")